# シュレディンガーブリッジ(wip)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# サンプルデータ生成（例として2つのガウス分布をブリッジする）
def generate_initial_data(num_samples):
    return np.random.normal(loc=-2, scale=1, size=(num_samples, 2))

def generate_final_data(num_samples):
    return np.random.normal(loc=2, scale=1, size=(num_samples, 2))

# ニューラルネットワークによるドリフト関数（スコア関数）の近似
class DriftNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DriftNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim + 1, hidden_dim), # +1 for time t
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x, t):
        # 時間 t を x と結合して入力
        # t が単一のスカラ値の場合、x のバッチサイズに合わせて拡張する
        # t_tensor = t.unsqueeze(1).expand_as(x[:, :1]) # 元のコード (エラーの原因)

        # 修正案1: t をバッチサイズに合わせて拡張 (最も推奨)
        # t が (1,) の形状の場合、unsqueeze(0) で (1, 1) にし、x のバッチ次元にexpand
        # t_tensor = t.unsqueeze(0).expand(x.shape[0], 1)

        # 修正案2: t を直接 x のバッチサイズに合わせて拡張 (より柔軟)
        t_tensor = t.expand(x.shape[0]).unsqueeze(1) # x のバッチサイズに拡張し、次元を追加

        input_tensor = torch.cat([x, t_tensor], dim=1)
        return self.net(input_tensor)

# SDEのシミュレーション（概念的な関数）
# 実際にはEuler-Maruyama法などを用いる
def simulate_sde(initial_samples, drift_net, num_timesteps, dt, sigma, direction='forward'):
    # ここにSDEの数値積分ロジックを実装
    # drift_net は学習されたドリフト関数（スコア関数）
    # sigma は拡散係数
    # direction は順方向（データからノイズ）か逆方向（ノイズからデータ）か

    # 簡略化のため、ここではダミーの軌道を返す
    trajectories = []
    current_samples = initial_samples
    for i in range(num_timesteps):
        t = i * dt
        # ドリフト項の計算
        # ドリフト項はスコア関数や学習された速度場に基づく
        drift = drift_net(torch.tensor(current_samples, dtype=torch.float32), torch.tensor(t, dtype=torch.float32))

        # 確率項の追加
        current_samples += drift.detach().numpy() * dt + sigma * np.sqrt(dt) * np.random.normal(size=current_samples.shape)

        trajectories.append(current_samples)
    return np.array(trajectories)


# IPFアルゴリズムのメインループ
def schrodinger_bridge_ipf(
    num_ipf_iterations,
    num_sde_timesteps,
    sde_dt,
    sde_sigma,
    num_samples,
    drift_net_fwd,  # 順方向のドリフトを学習するNN
    drift_net_bwd,  # 逆方向のドリフトを学習するNN
    optimizer_fwd,
    optimizer_bwd,
    num_epochs_per_ipf_step,
    batch_size
):
    # 初期データと最終データ
    data_0 = torch.tensor(generate_initial_data(num_samples), dtype=torch.float32)
    data_T = torch.tensor(generate_final_data(num_samples), dtype=torch.float32)

    # 最初の順方向プロセス（通常は参照のブラウン運動など）
    # ここでは単純なノーイズプロセスと仮定
    current_fwd_samples = data_0

    for ipf_iter in range(num_ipf_iterations):
        print(f"IPF Iteration {ipf_iter + 1}/{num_ipf_iterations}")

        # --- ステップ A: 逆方向の半ブリッジを学習 ---
        # (1) 現在の順方向プロセスからサンプルを生成
        # 実際には、current_fwd_samples から num_sde_timesteps にわたる軌道を生成し、
        # そのパス情報を用いて逆方向のドリフトを学習します。
        # ここでは簡略化のため、data_0 を始点としてランダムなノイズプロセスを仮定

        # SDEシミュレーションのダミー
        # actual_trajectories_fwd = simulate_sde(current_fwd_samples.numpy(), drift_net_fwd, num_sde_timesteps, sde_dt, sde_sigma, direction='forward')

        # 逆方向ドリフトネットのトレーニング
        for epoch in range(num_epochs_per_ipf_step):
            optimizer_bwd.zero_grad()
            # ここで、学習された順方向のドリフト（または現在の推測）と
            # 実際の目標終端分布（data_T）からのサンプルを用いて
            # 逆方向のドリフトを推定する損失を計算します。
            # これは、スコアマッチングの概念に基づくドリフトの推定に帰着します。

            # 簡略化された損失計算（実際とは異なります）
            # 例えば、data_T を終端とする SDE の逆過程のドリフトを学習
            # target_drift = ... (data_T とその周辺の確率密度勾配に依存)
            # predicted_drift = drift_net_bwd(some_samples_from_path, some_time)
            # loss_bwd = ((predicted_drift - target_drift)**2).mean()

            # ダミーの損失
            dummy_input = torch.randn(batch_size, 2) # ダミーの入力
            dummy_time = torch.rand(1) # ダミーの時間
            dummy_output = drift_net_bwd(dummy_input, dummy_time)
            loss_bwd = dummy_output.mean() # 実際には意味のある損失を設計

            loss_bwd.backward()
            optimizer_bwd.step()
        print(f"  Backward drift learned (Loss: {loss_bwd.item():.4f})")

        # --- ステップ B: 順方向の半ブリッジを学習 ---
        # (1) 以前に学習した逆方向プロセス（drift_net_bwd）と
        #     目標初期分布（data_0）からのサンプルを用いて、
        #     次の順方向プロセスの初期条件を設定。
        #     これにより、data_0 から data_T へ近づくパスが生成される。

        # 逆方向SDEシミュレーションのダミー
        # current_bwd_samples = simulate_sde(data_T.numpy(), drift_net_bwd, num_sde_timesteps, sde_dt, sde_sigma, direction='backward')

        # 順方向ドリフトネットのトレーニング
        for epoch in range(num_epochs_per_ipf_step):
            optimizer_fwd.zero_grad()
            # ここで、学習された逆方向のドリフトと
            # 実際の目標初期分布（data_0）からのサンプルを用いて
            # 順方向のドリフトを推定する損失を計算します。

            # ダミーの損失
            dummy_input = torch.randn(batch_size, 2) # ダミーの入力
            dummy_time = torch.rand(1) # ダミーの時間
            dummy_output = drift_net_fwd(dummy_input, dummy_time)
            loss_fwd = dummy_output.mean() # 実際には意味のある損失を設計

            loss_fwd.backward()
            optimizer_fwd.step()
        print(f"  Forward drift learned (Loss: {loss_fwd.item():.4f})")

        # 次のIPFイテレーションのための順方向サンプルを更新（概念的）
        # current_fwd_samples = サンプリングされた最終状態

    print("IPF training finished.")

    # 最終的な生成ステップ
    # 学習されたドリフトネット（通常はdrift_net_bwd）を使ってノイズからデータを生成
    print("\nGenerating samples from the learned bridge...")
    noise_samples = torch.randn(num_samples, 2) * sde_sigma # 終端のノイズ分布
    # generated_samples = simulate_sde(noise_samples.numpy(), drift_net_bwd, num_sde_timesteps, sde_dt, sde_sigma, direction='backward')

    # 最終的な生成のダミー
    # ここでの `torch.tensor(num_sde_timesteps * sde_dt, dtype=torch.float32)` は単一のスカラ値なので、
    # DriftNet の forward メソッドの修正によって正しく処理されるようになりました。
    final_generated_samples = noise_samples + 0.1 * drift_net_bwd(noise_samples, torch.tensor(num_sde_timesteps * sde_dt, dtype=torch.float32)).detach()
    print("Sample generation complete.")
    return final_generated_samples


# ハイパーパラメータの設定
input_dim = 2 # データの次元
hidden_dim = 64
output_dim = 2 # ドリフトの次元
num_ipf_iterations = 10
num_sde_timesteps = 100 # SDEシミュレーションの時間ステップ数
sde_dt = 1.0 / num_sde_timesteps
sde_sigma = 0.1 # 拡散係数
num_samples = 1000 # サンプル数
num_epochs_per_ipf_step = 50 # 各半ブリッジ学習のエポック数
batch_size = 128

# ドリフトネットワークの初期化
drift_net_fwd = DriftNet(input_dim, hidden_dim, output_dim)
drift_net_bwd = DriftNet(input_dim, hidden_dim, output_dim)

# オプティマイザの初期化
optimizer_fwd = optim.Adam(drift_net_fwd.parameters(), lr=1e-3)
optimizer_bwd = optim.Adam(drift_net_bwd.parameters(), lr=1e-3)

In [ ]:
# IPFアルゴリズムの実行
generated_samples = schrodinger_bridge_ipf(
    num_ipf_iterations,
    num_sde_timesteps,
    sde_dt,
    sde_sigma,
    num_samples,
    drift_net_fwd,
    drift_net_bwd,
    optimizer_fwd,
    optimizer_bwd,
    num_epochs_per_ipf_step,
    batch_size
)

print(f"\nShape of generated samples: {generated_samples.shape}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

def generate_initial_data(num_samples):
    return np.random.normal(loc=-2, scale=1, size=(num_samples, 2))

def generate_final_data(num_samples):
    return np.random.normal(loc=2, scale=1, size=(num_samples, 2))

class DriftNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DriftNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim + 1, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x, t):
        t_tensor = t.expand(x.shape[0]).unsqueeze(1)
        input_tensor = torch.cat([x, t_tensor], dim=1)
        return self.net(input_tensor)

def simulate_sde(initial_samples, drift_net, num_timesteps, dt, sigma, direction='forward'):
    trajectories = []
    current_samples = initial_samples
    for i in range(num_timesteps):
        t = i * dt
        trajectories.append(current_samples)
    return np.array(trajectories)

def schrodinger_bridge_ipf(
    num_ipf_iterations,
    num_sde_timesteps,
    sde_dt,
    sde_sigma,
    num_samples,
    drift_net_fwd,
    drift_net_bwd,
    optimizer_fwd,
    optimizer_bwd,
    num_epochs_per_ipf_step,
    batch_size
):
    data_0 = torch.tensor(generate_initial_data(num_samples), dtype=torch.float32)
    data_T = torch.tensor(generate_final_data(num_samples), dtype=torch.float32)

    current_fwd_samples = data_0

    for ipf_iter in range(num_ipf_iterations):
        print(f"IPF Iteration {ipf_iter + 1}/{num_ipf_iterations}")

        for epoch in range(num_epochs_per_ipf_step):
            optimizer_bwd.zero_grad()
            
            dummy_input = torch.randn(batch_size, 2)
            dummy_time = torch.rand(1)
            dummy_output = drift_net_bwd(dummy_input, dummy_time)
            loss_bwd = dummy_output.mean()

            loss_bwd.backward()
            optimizer_bwd.step()
        print(f"  Backward drift learned (Loss: {loss_bwd.item():.4f})")

        for epoch in range(num_epochs_per_ipf_step):
            optimizer_fwd.zero_grad()
            
            dummy_input = torch.randn(batch_size, 2)
            dummy_time = torch.rand(1)
            dummy_output = drift_net_fwd(dummy_input, dummy_time)
            loss_fwd = dummy_output.mean()

            loss_fwd.backward()
            optimizer_fwd.step()
        print(f"  Forward drift learned (Loss: {loss_fwd.item():.4f})")
        
    print("IPF training finished.")

    print("\nGenerating samples from the learned bridge...")
    noise_samples = torch.randn(num_samples, 2) * sde_sigma
    
    final_generated_samples = noise_samples + 0.1 * drift_net_bwd(noise_samples, torch.tensor(num_sde_timesteps * sde_dt, dtype=torch.float32)).detach()
    print("Sample generation complete.")
    return final_generated_samples

input_dim = 2
hidden_dim = 64
output_dim = 2
num_ipf_iterations = 10
num_sde_timesteps = 100
sde_dt = 1.0 / num_sde_timesteps
sde_sigma = 0.1
num_samples = 1000
num_epochs_per_ipf_step = 50
batch_size = 128

drift_net_fwd = DriftNet(input_dim, hidden_dim, output_dim)
drift_net_bwd = DriftNet(input_dim, hidden_dim, output_dim)

optimizer_fwd = optim.Adam(drift_net_fwd.parameters(), lr=1e-3)
optimizer_bwd = optim.Adam(drift_net_bwd.parameters(), lr=1e-3)

generated_samples = schrodinger_bridge_ipf(
    num_ipf_iterations,
    num_sde_timesteps,
    sde_dt,
    sde_sigma,
    num_samples,
    drift_net_fwd,
    drift_net_bwd,
    optimizer_fwd,
    optimizer_bwd,
    num_epochs_per_ipf_step,
    batch_size
)

print(f"\nShape of generated samples: {generated_samples.shape}")